# 08 · Build the presentation (consulting deck)

This notebook **is** the presentation. Every sentence, number and chart dataset on the Claude artifact (`docs/story.html`) is written here, one section per slide, and previewed below each cell as Markdown so you can read and debug it without opening the page.

```
notebooks 01–07  ──►  docs/results.json  ──►  this notebook (08)  ──►  docs/story.html  ──►  Claude artifact
   analysis            saved numbers           all wording + chart data      page             (published copy)
```

- **Numbers** come only from `docs/results.json` (notebooks 01–07) and `reference/assumptions.csv` (sourced outside inputs).
- **Wording** lives in the code cells as plain strings with light Markdown (`**bold**`, `*italic*`, `[link](url)`).
- **`site/story_template.html`** only lays out and draws; it has no text or numbers of its own.

**Storyline (answer first):**

| # | Slide | Feeds from |
|---|---|---|
| 0 | Executive summary: the answer | everything |
| 1 | Situation and complication → problem statement | outside sources, notebook 05 |
| 2 | Approach: three questions, three hypotheses | — |
| 3 | Data and privacy | notebooks 01–02 |
| 4 | A · Who keeps coming back | notebooks 03–04 |
| 5 | B · The lever and its price | notebook 05 |
| 6 | C · Where it pays for itself | notebook 06 |
| 7 | Recommendation | notebooks 04–06 |
| 8 | Risks, limitations, next steps | all |
| Appendix | Prescriber targeting, states, specialties | notebooks 05, 07 |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import html, json, re
import pandas as pd
from IPython.display import Markdown, display
from src.config import ROOT, RESULTS

R = json.loads(RESULTS.read_text())
C, P, M, K, T = R["cleaning"], R["profile"], R["models"], R["market"], R["targeting"]
V, L = R["value"], R["lever"]          # Part C (notebook 06) and Part B (notebook 05)
KS, AS, PA = R["kaggle_sample"], R["api_samples"], R["privacy_api"]
print("Sections available from notebooks 01–06:", list(R))

Sections available from notebooks 01–06: ['cleaning', 'sources', 'approvals', 'profile', 'models', 'market', 'targeting', 'kaggle_sample', 'api_samples', 'privacy_api', 'trend', 'lever', 'value']


## Helpers

`md()` turns the light Markdown used here into HTML for the page. `preview()` shows each section the way it reads on the page.

In [2]:
def md(s: str) -> str:
    """Inline Markdown -> HTML: **bold**, *italic*, `code`, [text](url)."""
    s = html.escape(s, quote=False)
    s = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", r'<a href="\2">\1</a>', s)
    s = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)
    s = re.sub(r"(?<![*\w])\*(?!\s)(.+?)(?<!\s)\*(?![*\w])", r"<i>\1</i>", s)
    s = re.sub(r"`(.+?)`", r'<span class="mono">\1</span>', s)
    return s

def pct(v, d=1): return f"{v:,.{d}f}%"
def num(v, d=0): return f"{v:,.{d}f}"

# Block constructors. Each keeps the Markdown source (for preview) and the HTML (for the page).
def p(text, cls=None):    return {"type": "p", "md": text, "html": md(text), **({"cls": cls} if cls else {})}
def h3(text):             return {"type": "h3", "md": text, "html": md(text)}
def say(text):            return {"type": "say", "md": text, "html": md(text)}
def bullets(items, title=None): return {"type": "list", "title": title, "md": items, "items": [md(i) for i in items]}
def text(title, body):    return {"type": "text", "title": title, "md": body, "html": md(body)}
def tiles(items):         return {"type": "tiles", "items": [{"value": v, "label": md(l), "src": s} for v, l, s in items]}
def cols(*blocks):        return {"type": "cols", "blocks": list(blocks)}
def table(columns, rows, highlight=None):
    """columns: list of (label, is_numeric)."""
    return {"type": "table", "columns": [{"label": c, "num": n} for c, n in columns], "rows": rows,
            **({"highlight": highlight} if highlight is not None else {})}
def chart(kind, title, data, opts, sub=None, legend=None):
    return {"type": "chart", "kind": kind, "title": title, "sub": sub, "legend": legend, "data": data, "opts": opts}

def preview(sec):
    """Render one section as Markdown so it can be read and checked inside the notebook."""
    out = [f"## {sec['title']}  \n*{sec['time']}*"]
    def walk(b):
        t = b["type"]
        if t in ("p", "h3"):   out.append(("### " if t == "h3" else "") + b["md"])
        elif t == "say":       out.append("> **Say:** " + b["md"])
        elif t == "callout":   out.append(f"> **{b.get('title') or ''}** " + b["md"])
        elif t == "text":      out.append(f"**{b['title']}**  \n{b['md']}")
        elif t == "list":      out.append(("**" + b["title"] + "**\n\n" if b.get("title") else "") + "\n".join("- " + i for i in b["md"]))
        elif t == "tiles":     out.append("\n".join(f"- **{i['value']}** {html.unescape(re.sub('<[^>]+>', '', i['label']))}" + (f" *({i['src']})*" if i.get("src") else "") for i in b["items"]))
        elif t == "cols":      [walk(x) for x in b["blocks"]]
        elif t == "table":     out.append(pd.DataFrame([[re.sub("<[^>]+>", "", str(v)) for v in r] for r in b["rows"]], columns=[c["label"] for c in b["columns"]]).to_markdown(index=False))
        elif t == "chart":
            out.append(f"**Chart · {b['title']}**" + (f" — *{b['sub']}*" if b.get("sub") else ""))
            d = b["data"]
            if b["kind"] in ("hbars",):     out.append(pd.DataFrame([{"label": r["label"], "value": r["value"]} for r in d]).to_markdown(index=False))
            elif b["kind"] == "pairbars":   out.append(pd.DataFrame([{"label": r["label"], **dict(zip(b["opts"]["names"], r["values"]))} for r in d]).to_markdown(index=False))
            elif b["kind"] == "lines":      out.append(pd.DataFrame({s["name"]: dict(s["pts"]) for s in d}).round(1).to_markdown())
            elif b["kind"] == "hgroup":     out.append(pd.DataFrame([{"group": r["label"], **dict(zip(b["opts"]["names"], r["values"]))} for r in d]).round(0).to_markdown(index=False))
            elif b["kind"] == "tornado":    out.append(pd.DataFrame([{"assumption": r["label"], "low": r["low"], "high": r["high"]} for r in d]).round(0).to_markdown(index=False))
            elif b["kind"] == "scatter":    out.append(pd.DataFrame([{"label": q["label"], "x": q["x"], "y": q["y"]} for q in d]).to_markdown(index=False))
        elif t == "issues":    out.append("\n".join(f"- **{q['k']}. {q['q']}** — {re.sub('<[^>]+>', '', q['how'])}" + (f" **[{q['verdict']}]**" if q.get("verdict") else "") for q in b["items"]))
        elif t == "refs":      out.append("Sources:\n" + "\n".join(f"{i+1}. {re.sub('<[^>]+>', '', r)}" for i, r in enumerate(b["items"])))
        elif t == "flow":      out.append("\n".join(f"- **{n['title']}** ({n['k']}): {n['text']}" for n in b["nodes"]) + "\n\n" + "\n".join(f"- *{j['key']}*: {j['text']}" for j in b["joins"]))
        elif t == "viewer":    out.append("Data viewer tabs: " + ", ".join(t_["label"] for t_ in b["tabs"]))
        elif t == "privacy":   out.append(f"**{b['title']}**  \n{b['md']}"); walk(b["table"]); out.append("\n".join("- " + x for x in b["bullets_md"]))
        elif t == "reco":      out.append("\n".join(f"{i+1}. **{r['k']}** — {r['md']}" for i, r in enumerate(b["items"])))
        elif t == "limits":    out.append("\n\n".join(f"**{c['title']}**\n" + "\n".join("- " + x for x in c["md"]) for c in b["cols"]))
    for b in sec["blocks"]:
        walk(b)
    display(Markdown("\n\n".join(out)))

SECTIONS = []

def callout(text, title=None, plain=False):
    return {"type": "callout", "title": title, "md": text, "html": md(text), **({"plain": True} if plain else {})}
def money(v, d=0):
    return ("−" if v < 0 else "") + "$" + f"{abs(v):,.{d}f}"

## Key numbers used across slides

Pulled once here so every slide quotes the same value. If a notebook changes, these update everywhere.

In [3]:
Q = V["qualifying"]
rule = {r["scenario"]: r for r in V["rule"]}
rule_half = {r["scenario"]: r for r in V["rule_half"]}
groups = {g["stays_group"]: g for g in V["groups"]}
SG_N, GL_N, SG_G, GL_G = "SGLT2 · negotiated (2026)", "GLP-1 · negotiated (2027)", "SGLT2 · gross 2024", "GLP-1 · gross 2024"
trend = {t["year"]: t for t in L["trend"]}
prior = {r["level"]: r for r in P["by"]["Prior hospital stays (past year)"]}
disc = {r["level"]: r for r in P["by"]["Discharged to"]}
best = max(M["comparison"], key=lambda r: r["ROC-AUC"])
KEY = {
    "qual_pct": Q["pct"], "qual_n": Q["n"], "share_readmits": Q["share_of_readmissions"], "qual_r30": Q["readmit_30"], "rest_r30": Q["readmit_30_rest"],
    "be_sglt2": V["breakeven"][SG_N], "be_glp1": V["breakeven"][GL_N], "be_sglt2_gross": V["breakeven"][SG_G],
    "net_per_pt": V["base_net"], "net_per_1000": rule[SG_N]["net / yr ($)"], "avoided_per_1000": rule[SG_N]["admissions avoided / yr"],
    "cpa_first": L["annual_cost"][SG_N] / 0.11, "two_plus_pct": Q["two_plus_pct"],
    "newer_claims_13": trend[2013]["newer_share_claims"], "newer_claims_24": trend[2024]["newer_share_claims"],
    "newer_cost_24": trend[2024]["newer_share_cost"], "spend_24": trend[2024]["cost_B"],
    "worst_2plus": min(V["stress"]["2+ prior stays"].values()), "neg_tests_1plus": sum(v < 0 for v in V["stress"]["1+ prior stays"].values()),
}
pd.Series(KEY).round(2).to_frame("value")

,value
qual_pct,11.70
qual_n,8195.00
share_readmits,20.10
qual_r30,15.40
rest_r30,8.10
be_sglt2,1.41
be_glp1,1.97
be_sglt2_gross,3.88
net_per_pt,1825.00
net_per_1000,213704.00


## Title and opening (0:00 – 0:30)

The title is the problem statement. The opener is personal; **rewrite it in your own words.**

In [4]:
HERO = {
    "eyebrow": "Case study · framed as a pharma client engagement · Adhvaith Vijay",
    "title": "Where does a GLP-1 or SGLT2 drug pay for itself?",
    "lede": "A value case for a diabetes-drug manufacturer: which diabetic patients avoid enough hospital stays on these drugs to cover their cost?",
    "opener": '"My dad has type 2 diabetes. When he was diagnosed, the options were mostly metformin, sulfonylureas and insulin. Today there are drugs that also protect the heart and kidneys, but they cost far more. So I asked: for which patients are they worth it?"',
    "opener_note": "Opening, in my own words. About 30 seconds.",
}
display(Markdown(f"# {HERO['title']}\n\n*{HERO['lede']}*\n\n> {HERO['opener']}"))
SECTIONS = []

# Where does a GLP-1 or SGLT2 drug pay for itself?

*A value case for a diabetes-drug manufacturer: which diabetic patients avoid enough hospital stays on these drugs to cover their cost?*

> "My dad has type 2 diabetes. When he was diagnosed, the options were mostly metformin, sulfonylureas and insulin. Today there are drugs that also protect the heart and kidneys, but they cost far more. So I asked: for which patients are they worth it?"

## Slide 0 · Executive summary (0:30 – 1:30)

The answer first, the way a consulting deck opens.

In [5]:
exec_sum = {"id": "summary", "nav": "Summary", "title": "The answer: target patients with another hospital stay in the past year", "time": "0:30 – 1:30", "blocks": [
    callout(f"At Medicare's negotiated prices, adding an SGLT2 drug **pays for itself through avoided hospital stays** for diabetic patients who've had **at least one other hospital stay in the past year**: {pct(KEY['qual_pct'])} of diabetic inpatients, who account for {pct(KEY['share_readmits'], 0)} of 30-day readmissions. For everyone else, it's a clinical choice, not a cost saving.", "The answer"),
    bullets([
        f"**Who.** {num(KEY['qual_n'])} of {num(C['patients'])} diabetic inpatients had another stay in the past year. They're readmitted within 30 days at {pct(KEY['qual_r30'])} vs. {pct(KEY['rest_r30'])} for first-time patients, and my readmission model ranks most of them in its top two risk deciles.",
        f"**What it costs.** A year of an SGLT2 drug is about {money(L['annual_cost'][SG_G])} at today's gross Medicare cost and {money(L['annual_cost'][SG_N])} at the negotiated 2026 price. Break-even is {KEY['be_sglt2']:.1f} hospital admissions a year at the negotiated price vs. {KEY['be_sglt2_gross']:.1f} at gross.",
        f"**What it's worth, and how sure.** Per 1,000 diabetic discharges, treating the qualifying patients avoids about {KEY['avoided_per_1000']:.0f} admissions a year and saves about {money(KEY['net_per_1000'])} net. That depends on the drug's effect, the price and how often these patients return. The robust core is the {pct(KEY['two_plus_pct'])} with **2+** prior stays: positive or within {money(abs(KEY['worst_2plus']))} of break-even in every stress test.",
    ], "Three supporting points"),
    say("I'll start with the answer, then show how I got there: who these patients are, what the drugs cost, and where the math works."),
]}
SECTIONS.append(exec_sum); preview(exec_sum)

## The answer: target patients with another hospital stay in the past year  
*0:30 – 1:30*

> **The answer** At Medicare's negotiated prices, adding an SGLT2 drug **pays for itself through avoided hospital stays** for diabetic patients who've had **at least one other hospital stay in the past year**: 11.7% of diabetic inpatients, who account for 20% of 30-day readmissions. For everyone else, it's a clinical choice, not a cost saving.

**Three supporting points**

- **Who.** 8,195 of 69,987 diabetic inpatients had another stay in the past year. They're readmitted within 30 days at 15.4% vs. 8.1% for first-time patients, and my readmission model ranks most of them in its top two risk deciles.
- **What it costs.** A year of an SGLT2 drug is about $6,481 at today's gross Medicare cost and $2,364 at the negotiated 2026 price. Break-even is 1.4 hospital admissions a year at the negotiated price vs. 3.9 at gross.
- **What it's worth, and how sure.** Per 1,000 diabetic discharges, treating the qualifying patients avoids about 32 admissions a year and saves about $213,704 net. That depends on the drug's effect, the price and how often these patients return. The robust core is the 3.4% with **2+** prior stays: positive or within $248 of break-even in every stress test.

> **Say:** I'll start with the answer, then show how I got there: who these patients are, what the drugs cost, and where the math works.

## Slide 1 · Situation, complication, problem statement (1:30 – 3:00)

The four outside facts carry sources; the Medicare figures come from notebook 05.

In [6]:
REFS = [
    "[CDC, National Diabetes Statistics Report: At a glance](https://www.cdc.gov/diabetes/php/data-research/index.html)",
    "[American Diabetes Association, Economic Costs of Diabetes in the U.S. in 2022](https://diabetes.org/about-us/statistics/cost-diabetes) (Parker et al., *Diabetes Care*)",
    "[Rubin & Shah, \"Predicting and Preventing Acute Care Re-Utilization by Patients with Diabetes,\" *Current Diabetes Reports*, 2021](https://pmc.ncbi.nlm.nih.gov/articles/PMC8418292)",
    "American Diabetes Association, Standards of Care in Diabetes, Section 9: Pharmacologic Approaches (*Diabetes Care* supplement)",
    "[GLP-1 RA / SGLT2i use after new ASCVD diagnosis, 2018–2022, *BMJ Open Diabetes Research & Care*, 2024](https://doaj.org/article/241fc9acf8aa4641a972e8d294236447)",
    "[CMS, Medicare negotiated prices for 2026](https://cms.gov/files/document/fact-sheet-negotiated-prices-initial-price-applicability-year-2026.pdf) and [2027 (semaglutide)](https://www.healthcaredive.com/news/medicare-price-negotiation-wegovy-ozempic-trelegy-2027/806578/)",
]
situation = {"id": "situation", "nav": "1 · Situation", "title": "1 · Expensive disease, effective drugs, and a pricing question", "time": "1:30 – 3:00", "blocks": [
    tiles([
        ("38.4M", "Americans have diabetes (11.6% of the population)", "CDC, 2021 data [1]"),
        ("$412.9B", "annual US cost of diabetes, $306.6B of it direct medical care", "ADA, 2022 [2]"),
        ("16–20%", "of hospitalized diabetic patients are readmitted within 30 days, about $20–25B a year", "Rubin & Shah, 2021 [3]"),
        (f"${KEY['spend_24']:.1f}B", "Medicare Part D spending on diabetes drugs in 2024, from my CMS pull", "CMS API, notebook 05"),
    ]),
    cols(
        text("Situation", "**GLP-1** (Ozempic, Trulicity, Mounjaro) and **SGLT2** drugs (Jardiance, Farxiga) do more than lower blood sugar: in large trials they cut heart attacks, heart-failure hospitalizations and kidney decline. Guidelines now recommend them for diabetic patients with heart or kidney disease regardless of blood sugar [4]."),
        text("Complication", f"They're expensive. They're {pct(KEY['newer_cost_24'], 0)} of Medicare diabetes drug spending but only {pct(KEY['newer_claims_24'], 0)} of prescriptions, and uptake among eligible patients is still low [5]. Payers ask whether the spending buys anything back. Medicare's new negotiated prices (Jardiance $197 a month from 2026, semaglutide $274 from 2027) change that math [6]."),
    ),
    callout("**For which diabetic patients does adding a GLP-1 or SGLT2 drug pay for itself by preventing hospital stays, and what is that worth to a payer?** The client is a manufacturer building its value case for Medicare plans and health systems.", "Problem statement"),
    say("The client doesn't need to hear that diabetes is expensive. It needs a defensible answer to a payer's question: where does paying for this drug come back as fewer hospital stays?"),
    {"type": "refs", "items": [md(r) for r in REFS]},
]}
SECTIONS.append(situation); preview(situation)

## 1 · Expensive disease, effective drugs, and a pricing question  
*1:30 – 3:00*

- **38.4M** Americans have diabetes (11.6% of the population) *(CDC, 2021 data [1])*
- **$412.9B** annual US cost of diabetes, $306.6B of it direct medical care *(ADA, 2022 [2])*
- **16–20%** of hospitalized diabetic patients are readmitted within 30 days, about $20–25B a year *(Rubin & Shah, 2021 [3])*
- **$59.9B** Medicare Part D spending on diabetes drugs in 2024, from my CMS pull *(CMS API, notebook 05)*

**Situation**  
**GLP-1** (Ozempic, Trulicity, Mounjaro) and **SGLT2** drugs (Jardiance, Farxiga) do more than lower blood sugar: in large trials they cut heart attacks, heart-failure hospitalizations and kidney decline. Guidelines now recommend them for diabetic patients with heart or kidney disease regardless of blood sugar [4].

**Complication**  
They're expensive. They're 76% of Medicare diabetes drug spending but only 33% of prescriptions, and uptake among eligible patients is still low [5]. Payers ask whether the spending buys anything back. Medicare's new negotiated prices (Jardiance $197 a month from 2026, semaglutide $274 from 2027) change that math [6].

> **Problem statement** **For which diabetic patients does adding a GLP-1 or SGLT2 drug pay for itself by preventing hospital stays, and what is that worth to a payer?** The client is a manufacturer building its value case for Medicare plans and health systems.

> **Say:** The client doesn't need to hear that diabetes is expensive. It needs a defensible answer to a payer's question: where does paying for this drug come back as fewer hospital stays?

Sources:
1. CDC, National Diabetes Statistics Report: At a glance
2. American Diabetes Association, Economic Costs of Diabetes in the U.S. in 2022 (Parker et al., Diabetes Care)
3. Rubin &amp; Shah, "Predicting and Preventing Acute Care Re-Utilization by Patients with Diabetes," Current Diabetes Reports, 2021
4. American Diabetes Association, Standards of Care in Diabetes, Section 9: Pharmacologic Approaches (Diabetes Care supplement)
5. GLP-1 RA / SGLT2i use after new ASCVD diagnosis, 2018–2022, BMJ Open Diabetes Research &amp; Care, 2024
6. CMS, Medicare negotiated prices for 2026 and 2027 (semaglutide)

## Slide 2 · Approach (3:00 – 4:00)

The problem broken into three questions. Each has a hypothesis, the data used, and the verdict from the analysis.

In [7]:
approach = {"id": "approach", "nav": "2 · Approach", "title": "2 · Three questions, each feeding the next", "time": "3:00 – 4:00", "blocks": [
    {"type": "issues", "items": [
        {"k": "A", "q": "Who keeps coming back to hospital?",
         "how": md("**Hypothesis:** a small, identifiable group of diabetic patients drives repeat admissions. **Data:** 101,766 hospital stays (Kaggle). **Methods:** SQL profiling, penalized regression, random forest, neural network, decision-tree segments."),
         "where": "Output → each patient's risk and hospital stays per year", "verdict": "Confirmed"},
        {"k": "B", "q": "What could change it, and what does it cost?",
         "how": md("**Hypothesis:** GLP-1 / SGLT2 drugs are the lever, but their price limits use. **Data:** CMS Part D 2013–2024 and 2020–2024 spending (my API pulls), CMS negotiated prices. **Methods:** same-source trend, cost per patient-year (gross vs. negotiated)."),
         "where": "Output → annual cost of treatment", "verdict": "Confirmed"},
        {"k": "C", "q": "Where does it pay for itself?",
         "how": md("**Hypothesis:** at negotiated prices, it pays for itself only for frequent hospital users. **Data:** A's admissions × B's costs × trial effect (EMPA-REG) × admission cost (AHRQ HCUP). **Methods:** break-even model, net value by group, sensitivity analysis."),
         "where": "Output → the targeting rule and its value", "verdict": "Confirmed, with limits"},
    ]},
    say("Each part's output is the next part's input: A tells us how often each patient is in hospital, B tells us what a year of the drug costs, and C combines them with the trial effect to find where it pays for itself."),
]}
SECTIONS.append(approach); preview(approach)

## 2 · Three questions, each feeding the next  
*3:00 – 4:00*

- **A. Who keeps coming back to hospital?** — Hypothesis: a small, identifiable group of diabetic patients drives repeat admissions. Data: 101,766 hospital stays (Kaggle). Methods: SQL profiling, penalized regression, random forest, neural network, decision-tree segments. **[Confirmed]**
- **B. What could change it, and what does it cost?** — Hypothesis: GLP-1 / SGLT2 drugs are the lever, but their price limits use. Data: CMS Part D 2013–2024 and 2020–2024 spending (my API pulls), CMS negotiated prices. Methods: same-source trend, cost per patient-year (gross vs. negotiated). **[Confirmed]**
- **C. Where does it pay for itself?** — Hypothesis: at negotiated prices, it pays for itself only for frequent hospital users. Data: A's admissions × B's costs × trial effect (EMPA-REG) × admission cost (AHRQ HCUP). Methods: break-even model, net value by group, sensitivity analysis. **[Confirmed, with limits]**

> **Say:** Each part's output is the next part's input: A tells us how often each patient is in hospital, B tells us what a year of the drug costs, and C combines them with the trial effect to find where it pays for itself.

## Slide 3 · Data and privacy (4:00 – 5:00)

Sources (row counts from notebook 02), how they connect, real rows from every source, and the privacy evidence. Show one data tab live; keep the rest for questions.

In [8]:
WHAT = {"Kaggle": "Patients, stays, drugs and readmissions, 1999–2008",
        "Provider & Drug": "Every prescriber's 2024 claims per diabetes drug, with specialty and state",
        "Spending": "Spending and claims per brand, 2020–2024", "national": "National claims and cost per drug, every year 2013–2024", "Geography": "Claims per state per drug, 2024",
        "CDC": "Adult diabetes and obesity rates per county", "RxClass": "Drug → class mapping (ATC)",
        "openFDA": "When each class was first approved"}
src_rows = [[s["name"], f'<span class="chip">{"My API pull" if s["kind"] == "API" else "Downloaded"}</span>', num(s["rows"]),
             next(v for k, v in WHAT.items() if k in s["name"])] for s in R["sources"]]

clean_chart = chart("hbars", "Cleaning the hospital file",
    [{"label": s["step"], "value": s["rows"], "color": "s1" if i == len(C["steps"]) - 1 else "s4",
      "tip": f"<b>{s['step']}</b><br>{num(s['rows'])} rows"} for i, s in enumerate(C["steps"])],
    {"labelW": 230, "max": 115000, "ticks": [0, 50000, 100000], "tickFmt": {"div": 1000, "suffix": "K"}, "valFmt": {}},
    sub="Each step and why it matters for the readmission rate")

In [9]:
# ---- data viewer: one tab per source, real rows saved by notebooks 01-02 ----
HEAD = {"Prscrbr_Type": "Specialty", "Prscrbr_State_Abrvtn": "State", "Brnd_Name": "Brand", "Gnrc_Name": "Generic", "Tot_Clms": "Claims",
        "Tot_Benes": "Patients", "Tot_Drug_Cst": "Drug cost ($)", "drug_class": "Class", "Tot_Spndng_2024": "Spending 2024 ($)",
        "Tot_Clms_2024": "Claims 2024", "Tot_Benes_2024": "Patients 2024", "Prscrbr_Geo_Desc": "State", "Tot_Prscrbrs": "Prescribers",
        "statedesc": "State", "locationname": "County", "totalpop18plus": "Adults", "diabetes_pct": "Diabetes %", "obesity_pct": "Obesity %",
        "atc_id": "ATC code", "atc_name": "ATC class", "ingredient": "Ingredient", "application": "FDA application",
        "first_approval": "First approved", "brand": "Brand"}
NUMERIC = re.compile(r"Clms|Benes|Cst|Spndng|Prscrbrs|pop18")

def fmt_cell(k, v):
    if v in ("nan", "None"):
        return "hidden by CMS (1–10)"                       # CMS blanks patient counts of 1-10
    if NUMERIC.search(k):
        n = float(v)
        return num(n, 2 if re.search("Cst|Spndng", k) and n < 1e6 else 0)
    return html.escape(v)

def sample_table(rows):
    keys = list(rows[0])
    return {"columns": [{"label": HEAD.get(k, k), "num": bool(NUMERIC.search(k))} for k in keys],
            "rows": [[fmt_cell(k, r[k]) for k in keys] for r in rows]}

tabs = [{
    "label": "Kaggle hospital stays",
    "kv": [["Source", md("Kaggle · Diabetes 130-US Hospitals for Years 1999–2008 (Strack et al., 2014), downloaded with `kagglehub`")],
           ["One row", "One hospital stay of a diabetic patient (1–14 days, with lab tests and medications recorded)"],
           ["Size", f"{num(KS['shape'][0])} stays × {KS['shape'][1]} columns, from {num(C['raw_patients'])} patients"]],
    "groups": [{"name": g, "cols": ", ".join(c[:6]) + (", …" if len(c) > 6 else "")} for g, c in KS["groups"].items()],
    "table": {"columns": [{"label": c, "num": False} for c in KS["columns_shown"]],
              "rows": [[html.escape(r[c]) for c in KS["columns_shown"]] for r in KS["rows"]]},
    "after": [{"html": md('6 random rows (seed 7), 13 of 50 columns. "?" = missing in the original file. Age comes only in 10-year bands. diag_1 is an ICD-9 code (996 = complications of a device or implant). readmitted: "<30" = within 30 days, ">30" = later, "NO" = not readmitted.')}],
}]
for a in AS:
    label = (a["name"].replace("CMS Part D ", "CMS · ").replace(" (2024)", "").replace(" (2020–2024)", "")
             .replace(" (2025 release)", "").replace(" county data", "").replace(" (ATC classes)", ""))
    after = []
    if a.get("raw_json"):
        after += [{"html": "The same Ozempic record as the API returns it (selected fields; every value arrives as text, so the notebook converts types):"},
                  {"code": html.escape(json.dumps(a["raw_json"], indent=2))}]
    if "Provider" in a["name"]:
        after.append({"html": md('Random rows, not top prescribers. The prescriber ID (NPI) is kept in the notebooks for counting prescribers but never shown. "Hidden by CMS" = fewer than 11 patients, which CMS blanks to protect privacy.')})
    tabs.append({"label": label,
                 "kv": [["Host", f'<span class="mono">{a["host"]}</span>'], ["Request", f'<pre class="code">{html.escape(a["request"])}</pre>'],
                        ["How I pulled it", html.escape(a["loop"])], ["One row", html.escape(a["one_row"])], ["Rows kept", num(a["total"])]],
                 "table": sample_table(a["rows"]), "after": after})
pd.DataFrame([{"tab": t["label"], "sample rows": len(t["table"]["rows"])} for t in tabs])

,tab,sample rows
0,Kaggle hospital stays,6
1,CMS · Prescribers by Provider & Drug,3
2,CMS · Spending by Drug,3
3,CMS · Prescribers by Geography & Drug,3
4,CDC PLACES,3
5,NLM RxClass,4
6,openFDA Drugs@FDA,4


In [10]:
# ---- privacy evidence: every number below was computed by a check in notebook 01 or 02 ----
KP = KS["privacy"]
left_out = [c for c in PA["prescriber_columns_offered"] if c not in PA["prescriber_columns_pulled"]]
name_city = [c for c in left_out if re.search("Name|City", c) and not re.search("Brnd|Gnrc", c)]
privacy_rows = [
    ["Kaggle hospital stays", "One de-identified hospital stay",
     f"No names, dates, addresses or ZIP codes ({len(KP['date_columns'])} date columns, {len(KP['name_address_zip_columns'])} name/address columns found). Age only in {len(KP['age_values'])} ten-year bands. Patient numbers are codes assigned during de-identification."],
    ["CMS prescribers by drug", "One prescriber's total claims for one drug",
     f"No patient rows. Smallest row: {PA['prescriber_min_claims_per_row']} claims (CMS drops anything under 11). Patient counts of 1–10 are blanked ({pct(PA['prescriber_benes_suppressed_pct'], 0)} of rows). Prescriber name and city not pulled ({len(name_city)} of {len(left_out)} unused columns)."],
    ["CMS spending by drug", "One brand, national totals", "National totals per brand; no people at all."],
    ["CMS claims by state", "One state × one brand", "Totals across every prescriber in the state."],
    ["CDC PLACES", "One county", "Survey-based percentages for a whole county, not records of people."],
    ["NLM RxClass, openFDA", "One drug ingredient", "Drug classes and approval dates; no people at all."],
]
PRIVACY_BULLETS = [
    "**Prescribers, not patients.** The CMS prescriber API also offers each prescriber's name and city. I didn't request those columns, and this page only shows totals by specialty, state and segment, never an individual prescriber.",
    "**Raw data stays on my laptop.** The downloaded files are excluded from the GitHub repo. The repo holds code, notebooks and aggregated results only.",
]
privacy = {"type": "privacy", "title": "No patient-identifying information anywhere",
           "md": "Nothing in this project can identify a patient. Every source is either de-identified research data or government open data published as totals, and everything was collected through official public APIs, not scraped.",
           "table": {"type": "table", "columns": [{"label": "Source", "num": False}, {"label": "What one row is", "num": False},
                                                  {"label": "Patient identifiers?", "num": False, "ok": True}, {"label": "Evidence (checked in the notebooks)", "num": False}],
                     "rows": [[f"<b>{s}</b>", r, "None", e] for s, r, e in privacy_rows]},
           "bullets_md": PRIVACY_BULLETS, "bullets": [md(b) for b in PRIVACY_BULLETS]}
privacy["html"] = md(privacy["md"])

In [11]:
data = {"id": "data", "nav": "3 · Data", "title": "3 · One downloaded dataset, seven API pulls, no patient identifiers", "time": "4:00 – 5:00", "blocks": [
    p("The hospital data describes the patients. My own API pulls (code in `src/apis.py`) supply what the drugs cost, how their use has changed, and how drugs map to classes. A small file of sourced outside inputs (`reference/assumptions.csv`) supplies the trial effect, the cost of a hospital stay and the negotiated prices."),
    table([("Source", False), ("How I got it", False), ("Rows", True), ("What it answers", False)], src_rows),
    clean_chart,
    {"type": "flow",
     "nodes": [{"k": "A · patients", "title": "Kaggle hospital stays", "text": "Who is readmitted; hospital stays in the past year", "downloaded": True},
               {"k": "shared classifier", "title": "NLM RxClass", "text": "Same drug-class rules for hospital and Medicare data (src/classes.py)"},
               {"k": "B · use and cost", "title": "CMS Part D", "text": "2013–2024 trend, spending, cost per 30-day fill"},
               {"k": "B · price", "title": "CMS negotiated prices", "text": "Jardiance, Farxiga 2026; semaglutide 2027"},
               {"k": "C · effect", "title": "Trial + HCUP", "text": "11% fewer hospitalizations; $15,200 per stay"}],
     "joins": [{"key": "admissions / year", "text": "From A (hospital stays in the past 12 months) into C."},
               {"key": "annual cost", "text": "From B (gross and negotiated prices) into C."},
               {"key": "drug class", "text": "One classifier for both data sets, so 'SGLT2' means the same drugs everywhere."}]},
    h3("What the data looks like"),
    {"type": "viewer", "tabs": tabs},
    privacy,
]}
SECTIONS.append(data); preview(data)

## 3 · One downloaded dataset, seven API pulls, no patient identifiers  
*4:00 – 5:00*

The hospital data describes the patients. My own API pulls (code in `src/apis.py`) supply what the drugs cost, how their use has changed, and how drugs map to classes. A small file of sourced outside inputs (`reference/assumptions.csv`) supplies the trial effect, the cost of a hospital stay and the negotiated prices.

| Source                                                          | How I got it   |      Rows | What it answers                                                            |
|:----------------------------------------------------------------|:---------------|----------:|:---------------------------------------------------------------------------|
| Kaggle · Diabetes 130-US Hospitals (1999–2008)                  | Downloaded     |   101,766 | Patients, stays, drugs and readmissions, 1999–2008                         |
| CMS Part D Prescribers by Provider & Drug (2024)                | My API pull    | 2,231,207 | Every prescriber's 2024 claims per diabetes drug, with specialty and state |
| CMS Part D Spending by Drug (2020–2024)                         | My API pull    |       142 | Spending and claims per brand, 2020–2024                                   |
| CMS Part D Prescribers by Geography & Drug (2024)               | My API pull    |     6,615 | Claims per state per drug, 2024                                            |
| CDC PLACES county data (2025 release)                           | My API pull    |     2,956 | Adult diabetes and obesity rates per county                                |
| NLM RxClass (ATC classes)                                       | My API pull    |        48 | Drug → class mapping (ATC)                                                 |
| CMS Part D Prescribers by Geography & Drug, national, 2013–2024 | My API pull    |     1,404 | National claims and cost per drug, every year 2013–2024                    |
| openFDA Drugs@FDA (approval dates)                              | My API pull    |        19 | When each class was first approved                                         |

**Chart · Cleaning the hospital file** — *Each step and why it matters for the readmission rate*

| label                             |   value |
|:----------------------------------|--------:|
| Raw hospital stays                |  101766 |
| Remove death / hospice discharges |   99343 |
| Keep first stay per patient       |   69990 |
| Drop unknown gender               |   69987 |

- **Kaggle hospital stays** (A · patients): Who is readmitted; hospital stays in the past year
- **NLM RxClass** (shared classifier): Same drug-class rules for hospital and Medicare data (src/classes.py)
- **CMS Part D** (B · use and cost): 2013–2024 trend, spending, cost per 30-day fill
- **CMS negotiated prices** (B · price): Jardiance, Farxiga 2026; semaglutide 2027
- **Trial + HCUP** (C · effect): 11% fewer hospitalizations; $15,200 per stay

- *admissions / year*: From A (hospital stays in the past 12 months) into C.
- *annual cost*: From B (gross and negotiated prices) into C.
- *drug class*: One classifier for both data sets, so 'SGLT2' means the same drugs everywhere.

### What the data looks like

Data viewer tabs: Kaggle hospital stays, CMS · Prescribers by Provider & Drug, CMS · Spending by Drug, CMS · Prescribers by Geography & Drug, CDC PLACES, NLM RxClass, openFDA Drugs@FDA

**No patient-identifying information anywhere**  
Nothing in this project can identify a patient. Every source is either de-identified research data or government open data published as totals, and everything was collected through official public APIs, not scraped.

| Source                  | What one row is                            | Patient identifiers?   | Evidence (checked in the notebooks)                                                                                                                                                     |
|:------------------------|:-------------------------------------------|:-----------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Kaggle hospital stays   | One de-identified hospital stay            | None                   | No names, dates, addresses or ZIP codes (0 date columns, 0 name/address columns found). Age only in 10 ten-year bands. Patient numbers are codes assigned during de-identification.     |
| CMS prescribers by drug | One prescriber's total claims for one drug | None                   | No patient rows. Smallest row: 11 claims (CMS drops anything under 11). Patient counts of 1–10 are blanked (64% of rows). Prescriber name and city not pulled (3 of 14 unused columns). |
| CMS spending by drug    | One brand, national totals                 | None                   | National totals per brand; no people at all.                                                                                                                                            |
| CMS claims by state     | One state × one brand                      | None                   | Totals across every prescriber in the state.                                                                                                                                            |
| CDC PLACES              | One county                                 | None                   | Survey-based percentages for a whole county, not records of people.                                                                                                                     |
| NLM RxClass, openFDA    | One drug ingredient                        | None                   | Drug classes and approval dates; no people at all.                                                                                                                                      |

- **Prescribers, not patients.** The CMS prescriber API also offers each prescriber's name and city. I didn't request those columns, and this page only shows totals by specialty, state and segment, never an individual prescriber.
- **Raw data stays on my laptop.** The downloaded files are excluded from the GitHub repo. The repo holds code, notebooks and aggregated results only.

## Slide 4 · A · Who keeps coming back (5:00 – 7:30)

Notebooks 03–04. The output that carries forward: hospital stays in the past year, by risk group.

In [12]:
p0, p3, home, snf = prior["0"], prior["3+"], disc["Home"], disc["Skilled nursing / care facility"]
nn = next(r for r in M["comparison"] if r["model"].startswith("Neural"))
rg = M["risk_groups"]
partA = {"id": "partA", "nav": "4 · A: Who", "title": f"4 · A: Repeat users are predictable: {pct(p3['readmit_pct'])} readmitted with 3+ prior stays vs. {pct(p0['readmit_pct'])} with none", "time": "5:00 – 7:30", "blocks": [
    tiles([(pct(P["overall_readmit"]), f"of {num(C['patients'])} diabetic patients were back within 30 days", None),
           (pct(KEY["qual_pct"]), "had at least one other hospital stay in the past year", None),
           (pct(P["a1c_tested_pct"], 0), "had their blood sugar (A1C) checked during the stay", None),
           (pct(100 - P["high_a1c_changed_pct"], 0), "of patients with a high A1C went home on unchanged diabetes meds", None)]),
    cols(
        chart("hbars", "Prior hospital stays drive readmission",
              [{"label": "None" if r["level"] == "0" else f"{r['level']} prior stay" + ("" if r["level"] == "1" else "s"), "value": r["readmit_pct"],
                "tip": f"<b>{r['level']} prior stays</b><br>{pct(r['readmit_pct'])} readmitted<br>{num(r['patients'])} patients ({pct(r['pct_of_patients'])})"} for r in P["by"]["Prior hospital stays (past year)"]],
              {"labelW": 120, "max": 30, "ticks": [0, 10, 20, 30], "tickFmt": {"suffix": "%"}, "valFmt": {"d": 1, "suffix": "%"}, "ref": P["overall_readmit"]},
              sub="% readmitted within 30 days · dashed line = all patients"),
        chart("hbars", "Hospital stays per year, by model risk decile",
              [{"label": "Highest risk" if g["risk_decile"] == 10 else "Lowest risk" if g["risk_decile"] == 1 else f"Decile {g['risk_decile']}",
                "value": g["admits_past_12m"], "color": "s2" if g["admits_past_12m"] >= KEY["be_sglt2"] else "s1",
                "tip": f"<b>Risk decile {g['risk_decile']}</b><br>{g['admits_past_12m']:.2f} stays in past 12 months<br>{pct(g['readmit_30_pct'])} readmitted within 30 days<br>{pct(g['prior_stay_pct'])} had a prior stay"} for g in reversed(rg)],
              {"labelW": 110, "rowH": 22, "max": 2.2, "ticks": [0, 1, 2], "tickFmt": {}, "valFmt": {"d": 2}, "ref": KEY["be_sglt2"]},
              sub=f"Stays in the past 12 months · dashed line = break-even at the negotiated SGLT2 price ({KEY['be_sglt2']:.2f})"),
    ),
    table([("Model", False), ("ROC-AUC", True), ("PR-AUC", True), ("Top-decile lift", True)],
          [[r["model"] + (' <span class="chip">best</span>' if r is best else ""), f"{r['ROC-AUC']:.3f}", f"{r['PR-AUC']:.3f}", f"{r['Top-decile lift']:.2f}×"] for r in M["comparison"]],
          highlight=M["comparison"].index(best)),
    say(f"History and destination predict a return far better than diagnosis: 3+ prior stays means {pct(p3['readmit_pct'])} readmitted, and a nursing-facility discharge {pct(snf['readmit_pct'])} vs. {pct(home['readmit_pct'])} home. Three models land at {min(r['ROC-AUC'] for r in M['comparison']):.2f}–{best['ROC-AUC']:.2f} AUC; the neural network adds nothing. What matters for the client is the right-hand chart: only the top risk deciles average more hospital stays a year than the drug needs to break even. And just {pct(P['a1c_tested_pct'], 0)} had an A1C test, so discharge is a missed moment to start treatment."),
]}
SECTIONS.append(partA); preview(partA)

## 4 · A: Repeat users are predictable: 26.5% readmitted with 3+ prior stays vs. 8.1% with none  
*5:00 – 7:30*

- **9.0%** of 69,987 diabetic patients were back within 30 days
- **11.7%** had at least one other hospital stay in the past year
- **18%** had their blood sugar (A1C) checked during the stay
- **40%** of patients with a high A1C went home on unchanged diabetes meds

**Chart · Prior hospital stays drive readmission** — *% readmitted within 30 days · dashed line = all patients*

| label          |   value |
|:---------------|--------:|
| None           |     8.1 |
| 1 prior stay   |    12.9 |
| 2 prior stays  |    18.5 |
| 3+ prior stays |    26.5 |

**Chart · Hospital stays per year, by model risk decile** — *Stays in the past 12 months · dashed line = break-even at the negotiated SGLT2 price (1.41)*

| label        |   value |
|:-------------|--------:|
| Highest risk |    1.77 |
| Decile 9     |    1.43 |
| Decile 8     |    1.24 |
| Decile 7     |    1.15 |
| Decile 6     |    1.09 |
| Decile 5     |    1.04 |
| Decile 4     |    1.02 |
| Decile 3     |    1.01 |
| Decile 2     |    1    |
| Lowest risk  |    1    |

| Model                   |   ROC-AUC |   PR-AUC | Top-decile lift   |
|:------------------------|----------:|---------:|:------------------|
| Penalized logistic (L1) |     0.643 |    0.156 | 2.01×             |
| Random forest best      |     0.656 |    0.163 | 2.32×             |
| Neural network (MLP)    |     0.635 |    0.15  | 2.00×             |

> **Say:** History and destination predict a return far better than diagnosis: 3+ prior stays means 26.5% readmitted, and a nursing-facility discharge 15.1% vs. 6.9% home. Three models land at 0.64–0.66 AUC; the neural network adds nothing. What matters for the client is the right-hand chart: only the top risk deciles average more hospital stays a year than the drug needs to break even. And just 18% had an A1C test, so discharge is a missed moment to start treatment.

## Slide 5 · B · The lever and its price (7:30 – 9:30)

Notebook 05: same-source trend 2013–2024, and the annual cost of treatment that carries into C.

In [13]:
bc = L["brand_costs"]
partB = {"id": "partB", "nav": "5 · B: Lever", "title": f"5 · B: The newer drugs went from {pct(KEY['newer_claims_13'])} to {pct(KEY['newer_claims_24'], 0)} of claims, and price is the barrier", "time": "7:30 – 9:30", "blocks": [
    chart("lines", "GLP-1 + SGLT2 share of Medicare diabetes drugs, 2013–2024 (same CMS file every year)",
          [{"name": "Share of spending", "color": "s2", "pts": [[t["year"], t["newer_share_cost"]] for t in L["trend"]], "end": pct(trend[2024]["newer_share_cost"], 0)},
           {"name": "Share of claims", "color": "s1", "pts": [[t["year"], t["newer_share_claims"]] for t in L["trend"]], "end": pct(trend[2024]["newer_share_claims"], 0)}],
          {"H": 250, "ymax": 100, "yticks": [0, 25, 50, 75, 100], "yFmt": {"suffix": "%"}, "xticks": [2013, 2016, 2019, 2022, 2024], "xFmt": {}, "tipFmt": {"d": 1, "suffix": "%"}},
          legend=[{"label": "Share of spending", "color": "s2"}, {"label": "Share of claims", "color": "s1"}]),
    chart("pairbars", "A year of treatment: gross 2024 Medicare cost vs. negotiated price",
          [{"label": b["Brnd_Name"] + f" ({b['cls']})", "values": [b["gross_year"], b["negotiated_year"] if b["negotiated_year"] == b["negotiated_year"] else 0],
            "tip": f"<b>{b['Brnd_Name']}</b><br>Gross 2024: ${b['gross_30d']:,.0f} per 30 days, ${b['gross_year']:,.0f} a year" + (f"<br>Negotiated: ${b['negotiated_30d']:,.0f} per 30 days" if b['negotiated_year'] == b['negotiated_year'] else "<br>No negotiated price yet")}
           for b in bc],
          {"max": 14000, "ticks": [0, 5000, 10000], "tickFmt": {"prefix": "$", "div": 1000, "suffix": "K"}, "valFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"},
           "colors": ["s4", "s1"], "names": ["Gross 2024", "Negotiated"]},
          legend=[{"label": "Gross 2024 (CMS: cost ÷ 30-day fills × 12)", "color": "s4"}, {"label": "Medicare negotiated price × 12", "color": "s1"}],
          sub="Trulicity and Mounjaro have no negotiated price yet"),
    say(f"In the 1999–2008 hospital data none of these drugs were used; the first GLP-1 arrived in 2005 and the first SGLT2 in 2013. From one consistent Medicare source, they went from {pct(KEY['newer_claims_13'])} of claims in 2013 to {pct(KEY['newer_claims_24'], 0)} in 2024, while taking {pct(KEY['newer_cost_24'], 0)} of spending. A year of an SGLT2 costs about {money(L['annual_cost'][SG_G])} gross today and {money(L['annual_cost'][SG_N])} at the negotiated price. That price is the input to the next slide."),
]}
SECTIONS.append(partB); preview(partB)

## 5 · B: The newer drugs went from 1.5% to 33% of claims, and price is the barrier  
*7:30 – 9:30*

**Chart · GLP-1 + SGLT2 share of Medicare diabetes drugs, 2013–2024 (same CMS file every year)**

|      |   Share of spending |   Share of claims |
|-----:|--------------------:|------------------:|
| 2013 |                 5.4 |               1.5 |
| 2014 |                 6.7 |               2.1 |
| 2015 |                 9.3 |               3.2 |
| 2016 |                12.5 |               4.3 |
| 2017 |                16.5 |               5.5 |
| 2018 |                20.9 |               7   |
| 2019 |                26.6 |               9.2 |
| 2020 |                33.6 |              11.7 |
| 2021 |                41.5 |              15.2 |
| 2022 |                50.9 |              20   |
| 2023 |                60.9 |              27.4 |
| 2024 |                76   |              33.3 |

**Chart · A year of treatment: gross 2024 Medicare cost vs. negotiated price** — *Trulicity and Mounjaro have no negotiated price yet*

| label             |   Gross 2024 |   Negotiated |
|:------------------|-------------:|-------------:|
| Ozempic (GLP-1)   |        10840 |         3288 |
| Jardiance (SGLT2) |         6491 |         2364 |
| Mounjaro (GLP-1)  |        12475 |            0 |
| Trulicity (GLP-1) |        11381 |            0 |
| Farxiga (SGLT2)   |         6460 |         2142 |
| Rybelsus (GLP-1)  |        10989 |         3288 |

> **Say:** In the 1999–2008 hospital data none of these drugs were used; the first GLP-1 arrived in 2005 and the first SGLT2 in 2013. From one consistent Medicare source, they went from 1.5% of claims in 2013 to 33% in 2024, while taking 76% of spending. A year of an SGLT2 costs about $6,481 gross today and $2,364 at the negotiated price. That price is the input to the next slide.

## Slide 6 · C · Where it pays for itself (9:30 – 12:30)

Notebook 06: break-even, net value by group, the rule per 1,000 discharges, sensitivity.

In [14]:
ORDER = [SG_N, GL_N, SG_G, GL_G]
grp_rows = [{"label": g["stays_group"], "values": [g[k] for k in ORDER],
             "tip": f"<b>{g['stays_group']}</b> ({pct(g['share_pct'])} of patients)<br>{g['admits_past_12m']:.1f} stays in past 12 months<br>" + "<br>".join(f"{k}: {money(g[k])}" for k in ORDER)}
            for g in V["groups"]]
tor = V["tornado"]
partC = {"id": "partC", "nav": "6 · C: Value", "title": f"6 · C: At negotiated prices it pays for itself once a patient has another stay in the past year", "time": "9:30 – 12:30", "blocks": [
    p(f"**The model, per patient-year:** admissions avoided = expected admissions × 11% (EMPA-REG trial); savings = avoided × $15,200 (AHRQ HCUP); net value = savings − annual drug cost. Expected admissions = stays in the past 12 months from Part A. Only hospital savings are counted, so this is conservative."),
    table([("Price scenario", False), ("Annual drug cost", True), ("Break-even admissions / year", True)],
          [[k, money(L["annual_cost"][k]), f"{V['breakeven'][k]:.2f}"] for k in ORDER]),
    chart("hgroup", "Net value per patient-year, by hospital stays in the past year", grp_rows,
          {"labelW": 120, "dom": [-12000, 7000], "ticks": [-10000, -5000, 0, 5000], "tickFmt": {"prefix": "$", "div": 1000, "suffix": "K"},
           "valFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"}, "colors": ["s1", "s3", "s4", "s2"], "names": ORDER},
          legend=[{"label": k, "color": c} for k, c in zip(ORDER, ["s1", "s3", "s4", "s2"])],
          sub="Positive = the drug pays for itself through avoided hospital stays"),
    h3("The rule, sized per 1,000 diabetic discharges"),
    table([("Price scenario", False), ("Qualifying patients", True), ("Admissions avoided / yr", True), ("Drug cost / yr", True), ("Hospital savings / yr", True), ("Net / yr", True), ("Cost per admission avoided", True)],
          [[k, num(rule[k]["qualifying per 1,000 discharges"]), num(rule[k]["admissions avoided / yr"]), money(rule[k]["drug cost / yr ($)"]),
            money(rule[k]["hospital savings / yr ($)"]), f"<b>{money(rule[k]['net / yr ($)'])}</b>", money(rule[k]["cost per admission avoided ($)"])] for k in ORDER]),
    chart("tornado", "How sure are we? Net value per qualifying patient-year at the negotiated SGLT2 price",
          [{"label": t["assumption"], "low": t["low"], "high": t["high"], "tip": f"<b>{t['assumption']}</b><br>low: {money(t['low'])}<br>high: {money(t['high'])}"} for t in tor],
          {"labelW": 260, "dom": [-3000, 5000], "ticks": [-2000, 0, 2000, 4000], "tickFmt": {"prefix": "$", "div": 1000, "d": 1, "suffix": "K"}, "base": V["base_net"]},
          sub="Each bar moves one assumption between its low and high value; orange = can turn negative"),
    say(f"Break-even at the negotiated SGLT2 price is {KEY['be_sglt2']:.1f} admissions a year. First-time patients have one, so for them the drug costs about {money(KEY['cpa_first'])} per admission avoided. Patients with another stay in the past year clear it: about {money(KEY['net_per_pt'])} net per patient-year, or {money(KEY['net_per_1000'])} per 1,000 discharges. At gross prices only SGLT2 for patients with 3+ prior stays pays off; GLP-1 never does. The biggest uncertainties are the size of the effect and price. If future stays are only half the past year, the case holds only for the {pct(KEY['two_plus_pct'])} with 2+ prior stays."),
]}
SECTIONS.append(partC); preview(partC)

## 6 · C: At negotiated prices it pays for itself once a patient has another stay in the past year  
*9:30 – 12:30*

**The model, per patient-year:** admissions avoided = expected admissions × 11% (EMPA-REG trial); savings = avoided × $15,200 (AHRQ HCUP); net value = savings − annual drug cost. Expected admissions = stays in the past 12 months from Part A. Only hospital savings are counted, so this is conservative.

| Price scenario            | Annual drug cost   |   Break-even admissions / year |
|:--------------------------|:-------------------|-------------------------------:|
| SGLT2 · negotiated (2026) | $2,364             |                           1.41 |
| GLP-1 · negotiated (2027) | $3,288             |                           1.97 |
| SGLT2 · gross 2024        | $6,481             |                           3.88 |
| GLP-1 · gross 2024        | $11,313            |                           6.77 |

**Chart · Net value per patient-year, by hospital stays in the past year** — *Positive = the drug pays for itself through avoided hospital stays*

| group          |   SGLT2 · negotiated (2026) |   GLP-1 · negotiated (2027) |   SGLT2 · gross 2024 |   GLP-1 · gross 2024 |
|:---------------|----------------------------:|----------------------------:|---------------------:|---------------------:|
| This stay only |                        -692 |                       -1616 |                -4809 |                -9641 |
| 1 prior stay   |                         980 |                          56 |                -3137 |                -7969 |
| 2 prior stays  |                        2652 |                        1728 |                -1465 |                -6297 |
| 3+ prior stays |                        5908 |                        4984 |                 1791 |                -3041 |

### The rule, sized per 1,000 diabetic discharges

| Price scenario            |   Qualifying patients |   Admissions avoided / yr | Drug cost / yr   | Hospital savings / yr   | Net / yr   | Cost per admission avoided   |
|:--------------------------|----------------------:|--------------------------:|:-----------------|:------------------------|:-----------|:-----------------------------|
| SGLT2 · negotiated (2026) |                   117 |                        32 | $276,808         | $490,513                | $213,704   | $8,578                       |
| GLP-1 · negotiated (2027) |                   117 |                        32 | $385,002         | $490,513                | $105,510   | $11,930                      |
| SGLT2 · gross 2024        |                   117 |                        32 | $758,881         | $490,513                | −$268,368  | $23,516                      |
| GLP-1 · gross 2024        |                   117 |                        32 | $1,324,675       | $490,513                | −$834,163  | $41,049                      |

**Chart · How sure are we? Net value per qualifying patient-year at the negotiated SGLT2 price** — *Each bar moves one assumption between its low and high value; orange = can turn negative*

| assumption                                  |   low |   high |
|:--------------------------------------------|------:|-------:|
| Hospitalization reduction (4% – 18%)        |  -841 |   4491 |
| Price (negotiated – gross 2024)             | -2292 |   1825 |
| Cost per admission ($10,900 – $24,200)      |   640 |   4305 |
| Future admissions (50% – 100% of past year) |  -269 |   1825 |

> **Say:** Break-even at the negotiated SGLT2 price is 1.4 admissions a year. First-time patients have one, so for them the drug costs about $21,491 per admission avoided. Patients with another stay in the past year clear it: about $1,825 net per patient-year, or $213,704 per 1,000 discharges. At gross prices only SGLT2 for patients with 3+ prior stays pays off; GLP-1 never does. The biggest uncertainties are the size of the effect and price. If future stays are only half the past year, the case holds only for the 3.4% with 2+ prior stays.

## Slide 7 · Recommendation (12:30 – 14:00)

In [15]:
RECO = [
    ("Lead the value case with one patient group", f"Build the payer story around diabetic patients with **another hospital stay in the past year** ({pct(KEY['qual_pct'])} of inpatients, {pct(KEY['share_readmits'], 0)} of 30-day readmissions). It's the group where the drug pays for itself at negotiated prices, and it's simple to find in any claims system."),
    ("Start treatment at discharge", f"Only {pct(P['a1c_tested_pct'], 0)} of diabetic inpatients had an A1C test, and {pct(100 - P['high_a1c_changed_pct'], 0)} of those with a high result went home on unchanged medication. Partner with health systems on discharge protocols that flag repeat-stay diabetic patients for a GLP-1 / SGLT2 review."),
    ("Price to the negotiated level, not list", f"At gross 2024 cost the case fails for nearly everyone; at the negotiated price it works for the target group. Contracting with Medicare Advantage and commercial plans should aim near the negotiated price, where break-even falls from {KEY['be_sglt2_gross']:.1f} to {KEY['be_sglt2']:.1f} admissions a year."),
    ("Prove it with a pilot", f"Run a 12-month pilot with one health system: treat qualifying patients, compare hospital stays with matched patients who weren't treated, and start with the {pct(KEY['two_plus_pct'])} with 2+ prior stays, where the case stays at or near break-even in every stress test."),
]
reco = {"id": "reco", "nav": "7 · Recommendation", "title": "7 · Recommendation", "time": "12:30 – 14:00", "blocks": [
    {"type": "reco", "items": [{"k": k, "md": v, "html": md(v)} for k, v in RECO]},
    say("Four actions, in order: who to lead with, when to reach them, what price makes the case work, and how to prove it."),
]}
SECTIONS.append(reco); preview(reco)

## 7 · Recommendation  
*12:30 – 14:00*

1. **Lead the value case with one patient group** — Build the payer story around diabetic patients with **another hospital stay in the past year** (11.7% of inpatients, 20% of 30-day readmissions). It's the group where the drug pays for itself at negotiated prices, and it's simple to find in any claims system.
2. **Start treatment at discharge** — Only 18% of diabetic inpatients had an A1C test, and 40% of those with a high result went home on unchanged medication. Partner with health systems on discharge protocols that flag repeat-stay diabetic patients for a GLP-1 / SGLT2 review.
3. **Price to the negotiated level, not list** — At gross 2024 cost the case fails for nearly everyone; at the negotiated price it works for the target group. Contracting with Medicare Advantage and commercial plans should aim near the negotiated price, where break-even falls from 3.9 to 1.4 admissions a year.
4. **Prove it with a pilot** — Run a 12-month pilot with one health system: treat qualifying patients, compare hospital stays with matched patients who weren't treated, and start with the 3.4% with 2+ prior stays, where the case stays at or near break-even in every stress test.

> **Say:** Four actions, in order: who to lead with, when to reach them, what price makes the case work, and how to prove it.

## Slide 8 · Risks, limitations, next steps (14:00 – 15:00)

In [16]:
LIMITS = [
    "The hospital data is 1999–2008 and observational; today's patients and hospital practices differ.",
    "The 11% effect comes from a trial in patients with heart disease; real-world adherence is lower.",
    "Next year's admissions are assumed to match the past year; frequent users drift toward average (tested at 50%).",
    "Only hospital savings are counted, not deaths, kidney failure or quality of life, so value is understated.",
    "Gross Medicare cost is before confidential rebates; the true net price sits between gross and negotiated.",
]
NEXT = [
    "Re-estimate admissions with recent Medicare claims (e.g. CMS research files) instead of 1999–2008 data.",
    "Add the heart-failure and kidney benefits, which are larger than the all-cause hospitalization effect.",
    "Survival analysis of time to readmission, by treatment, once patient-level drug data is available.",
    "Size the national opportunity using the number of diabetic discharges per year.",
]
risks = {"id": "risks", "nav": "8 · Risks", "title": "8 · Risks, limitations and next steps", "time": "14:00 – 15:00", "blocks": [
    {"type": "limits", "cols": [{"title": "Limitations", "md": LIMITS, "items": [md(x) for x in LIMITS]},
                                {"title": "Next steps", "md": NEXT, "items": [md(x) for x in NEXT]}]},
    say("The direction holds up; the exact dollar figures depend on assumptions I've laid out and tested. The next step is the same model on recent claims data."),
]}
SECTIONS.append(risks); preview(risks)

## 8 · Risks, limitations and next steps  
*14:00 – 15:00*

**Limitations**
- The hospital data is 1999–2008 and observational; today's patients and hospital practices differ.
- The 11% effect comes from a trial in patients with heart disease; real-world adherence is lower.
- Next year's admissions are assumed to match the past year; frequent users drift toward average (tested at 50%).
- Only hospital savings are counted, not deaths, kidney failure or quality of life, so value is understated.
- Gross Medicare cost is before confidential rebates; the true net price sits between gross and negotiated.

**Next steps**
- Re-estimate admissions with recent Medicare claims (e.g. CMS research files) instead of 1999–2008 data.
- Add the heart-failure and kidney benefits, which are larger than the all-cause hospitalization effect.
- Survival analysis of time to readmission, by treatment, once patient-level drug data is available.
- Size the national opportunity using the number of diabetic discharges per year.

> **Say:** The direction holds up; the exact dollar figures depend on assumptions I've laid out and tested. The next step is the same model on recent claims data.

## Appendix (not timed): prescriber targeting, states, specialties

Earlier analyses kept for questions. Numbers from notebooks 05 and 07.

In [17]:
seg_head = {s["segment"]: s for s in T["segment_headroom"]}
specs = [s for s in K["specialties"] if s["diabetes_claims"] >= 350000]
SPEC_LBL = {"Internal Medicine": (0, 26, "middle"), "Family Practice": (16, 4, "start"), "Nurse Practitioner": (8, -26, "start"),
            "Physician Assistant": (0, 26, "middle"), "Endocrinology": (-4, -14, "end"), "Nephrology": (-12, 4, "end")}
PRIMARY = re.compile("Family|Internal|Nurse|Physician Assistant|General")
spec_pts = []
for s in specs:
    r = 4 + s["prescribers"] ** 0.5 / 18
    dx, dy, anchor = SPEC_LBL.get(s["specialty"], (r + 4, 4, "start"))
    spec_pts.append({"x": s["share_of_all_claims_pct"], "y": s["advanced_pct"], "r": r, "label": s["specialty"], "showLabel": True, "dx": dx, "dy": dy, "anchor": anchor,
                     "color": "s3" if s["specialty"] == "Endocrinology" else "s2" if PRIMARY.search(s["specialty"]) else "s1",
                     "tip": f"<b>{s['specialty']}</b><br>{num(s['prescribers'])} prescribers · {pct(s['share_of_all_claims_pct'])} of claims<br>GLP-1 {pct(s['glp1_pct'])} · SGLT2 {pct(s['sglt2_pct'])}"})
appx = {"id": "appendix", "nav": "Appendix", "title": "Appendix · Who would carry this out: prescribers", "time": "for Q&A", "appendix_start": True, "blocks": [
    p("Earlier analysis of the 2024 Medicare prescriber file. Not part of the main argument, but it answers \"which doctors would act on the recommendation?\""),
    chart("scatter", "Claim volume vs. newer-drug share, by specialty", spec_pts,
          {"xlog": True, "xdom": [0.25, 45], "ydom": [10, 100], "xticks": [0.3, 1, 3, 10, 30], "yticks": [20, 40, 60, 80, 100],
           "xFmt": {"d": 1, "suffix": "%"}, "yFmt": {"suffix": "%"}, "xlabel": "Share of all Medicare diabetes claims (log scale)", "refY": K["national_advanced_pct"], "H": 380},
          sub="Primary care (orange) writes most diabetes claims and trails endocrinology (green); cardiology and nephrology use SGLT2 for heart and kidney disease"),
    table([("Prescriber segment (k-means)", False), ("Prescribers", True), ("Share of claims", True), ("GLP-1 + SGLT2", True), ("Share of headroom", True)],
          [[s["segment"], num(s["prescribers"]), pct(s["claims_share_pct"]), pct(s["advanced_pct"]), pct(seg_head[s["segment"]]["headroom_share_pct"])]
           for s in sorted(T["segments"], key=lambda s: s["advanced_pct"])]),
    p(f"Headroom = how far a prescriber sits below similar prescribers, times their volume. The top 20% of prescribers hold {pct(T['top20_share'], 0)} of it. Caveat: headroom can't see each doctor's patient mix, so it's a place to start conversations, not proof of under-treatment."),
]}
SECTIONS.append(appx); preview(appx)

## Appendix · Who would carry this out: prescribers  
*for Q&A*

Earlier analysis of the 2024 Medicare prescriber file. Not part of the main argument, but it answers "which doctors would act on the recommendation?"

**Chart · Claim volume vs. newer-drug share, by specialty** — *Primary care (orange) writes most diabetes claims and trails endocrinology (green); cardiology and nephrology use SGLT2 for heart and kidney disease*

| label               |    x |    y |
|:--------------------|-----:|-----:|
| Family Practice     | 30.7 | 31   |
| Internal Medicine   | 27.8 | 31.1 |
| Nurse Practitioner  | 18.2 | 36.9 |
| Endocrinology       |  9   | 40   |
| Physician Assistant |  5.9 | 38   |
| General Practice    |  2.3 | 19.5 |
| Cardiology          |  1.6 | 82.6 |
| Nephrology          |  1   | 79.7 |
| Geriatric Medicine  |  0.5 | 23.3 |
| Pharmacist          |  0.5 | 37.6 |
| Hospitalist         |  0.4 | 31   |

| Prescriber segment (k-means)   |   Prescribers | Share of claims   | GLP-1 + SGLT2   | Share of headroom   |
|:-------------------------------|--------------:|:------------------|:----------------|:--------------------|
| Traditional orals              |        81,198 | 42.6%             | 23.1%           | 77.5%               |
| Insulin-heavy                  |        28,637 | 27.8%             | 29.4%           | 18.2%               |
| Newer-drug adopters            |        64,438 | 27.3%             | 47.0%           | 3.7%                |
| SGLT2-only (heart / kidney)    |         9,761 | 2.4%              | 96.1%           | 0.5%                |

Headroom = how far a prescriber sits below similar prescribers, times their volume. The top 20% of prescribers hold 76% of it. Caveat: headroom can't see each doctor's patient mix, so it's a place to start conversations, not proof of under-treatment.

## Build the page

Checks, then inject everything into `site/story_template.html` and write `docs/story.html`.

In [18]:
FOOTER = ["Built from eight Jupyter notebooks in <span class=\"mono\">avaitt/personal-project</span>. Every number is read from the notebooks' saved results or the sourced assumptions file.",
          "Data: Kaggle Diabetes 130-US Hospitals (Strack et al., 2014) · CMS Medicare Part D (2013–2024) · CMS negotiated prices · EMPA-REG OUTCOME · AHRQ HCUP · CDC PLACES · NLM RxClass · openFDA."]
PAGE = {**HERO, "sections": SECTIONS, "footer": FOOTER, "appendix_note": "Supporting analyses kept for questions. Not part of the 15 minutes."}
blob = json.dumps(PAGE, ensure_ascii=False)
assert all(s["title"] and s["time"] and s["blocks"] for s in SECTIONS)
assert "NaN" not in blob, "a missing value reached the page"
print("Section times:", [s["time"] for s in SECTIONS])
template = (ROOT / "site" / "story_template.html").read_text()
assert template.count("/*__PAGE__*/{}") == 1
out = ROOT / "docs" / "story.html"
out.write_text(template.replace("/*__PAGE__*/{}", json.dumps(PAGE, ensure_ascii=False, separators=(",", ":"))))
print(f"wrote {out.relative_to(ROOT)} ({out.stat().st_size / 1024:.0f} KB, {len(SECTIONS)} sections)")

Section times: ['0:30 – 1:30', '1:30 – 3:00', '3:00 – 4:00', '4:00 – 5:00', '5:00 – 7:30', '7:30 – 9:30', '9:30 – 12:30', '12:30 – 14:00', '14:00 – 15:00', 'for Q&A']
wrote docs/story.html (73 KB, 10 sections)
